# importing libraries


In [54]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [47]:
df = pd.read_csv("House Price Prediction Dataset.csv")
df = df.drop(columns=["Id"])

In [48]:
df.head()
# print(df.info)
# print(df.shape)

,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price
0,1360,5,4,3,1970,Downtown,Excellent,No,149919
1,4272,5,4,3,1958,Downtown,Excellent,No,424998
2,3592,2,2,3,1938,Downtown,Good,No,266746
3,966,4,2,2,1902,Suburban,Fair,Yes,244020
4,4926,1,4,2,1975,Downtown,Fair,Yes,636056


# seprate features and target

In [55]:
X = df.drop(columns=["Price"])
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# encoding the values for garage
garage_enc = OrdinalEncoder(categories=[["No", "Yes"]])
X_train["Garage"] = garage_enc.fit_transform(X_train[["Garage"]])
X_test["Garage"] = garage_enc.transform(X_test[["Garage"]])

# encoding the values for condition
condition_enc = OrdinalEncoder(categories=[["Poor", "Fair", "Good", "Excellent"]])
X_train["Condition"] = condition_enc.fit_transform(X_train[["Condition"]])
X_test["Condition"] = condition_enc.transform(X_test[["Condition"]])

# encoding values for location
loc_enc = OneHotEncoder(drop="first", sparse_output=False)
loc_enc.set_output(transform="pandas")

train_loc = loc_enc.fit_transform(X_train[["Location"]])
test_loc = loc_enc.transform(X_test[["Location"]])

X_train = pd.concat([X_train.drop(columns="Location"), train_loc], axis=1)
X_test = pd.concat([X_test.drop(columns="Location"), test_loc], axis=1)

# scalling the values of area bedroom,bathroom,flors and yearbuilt.
num_cols = ["Area", "Bedrooms", "Bathrooms", "Floors", "YearBuilt"]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train.head()

,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Condition,Garage,Location_Rural,Location_Suburban,Location_Urban
968,1.314081,0.685758,1.316088,1.242260,-0.808673,3.0,0.0,0.0,0.0,1.0
240,-1.315960,-0.014889,0.414272,-1.236064,0.221883,2.0,0.0,0.0,0.0,0.0
819,-1.039194,-0.014889,1.316088,-1.236064,0.862499,2.0,1.0,0.0,0.0,1.0
692,-0.088966,-0.715536,0.414272,-1.236064,0.277589,0.0,1.0,1.0,0.0,0.0
420,0.393836,-0.715536,1.316088,-1.236064,0.528264,3.0,1.0,1.0,0.0,0.0


# training a baseline model for benchmarks

In [56]:
from sklearn.dummy import  DummyRegressor
from sklearn.metrics import  mean_absolute_error,r2_score

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train,y_train)

pred = baseline.predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred))
print("R2 :", r2_score(y_test, pred))

MAE: 242480.2015
R2 : -0.0007164349053967456


# training a real model

In [59]:
from sklearn.linear_model import  LinearRegression

model = LinearRegression()
model.fit(X_train,y_train)
pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred))
print("R2 :", r2_score(y_test, pred))

MAE: 243453.00113858172
R2 : -0.00814309997438234


In [60]:
# check for overfitting
print("Train R2:", r2_score(y_train, model.predict(X_train)))
print("Test R2 :", r2_score(y_test, pred))

Train R2: 0.007937984359026595
Test R2 : -0.00814309997438234


In [61]:
# exporting the model
import joblib

joblib.dump(
    {
        "model": model,
        "garage": garage_enc,
        "condition": condition_enc,
        "location": loc_enc,
        "scaler": scaler,
    },
    "house_price_bundle.joblib",
)

['house_price_bundle.joblib']

In [62]:
# parameters
print(model.coef_)  # one weight per feature
print(model.intercept_)  # the single intercept

n_params = model.coef_.size + 1
print("Total parameters:", n_params)

[  -420.41094664    219.44177056 -11107.09412326  19188.60250172
   3811.41865394  -4848.24020812   2669.26509857   2265.04131137
  12514.26227327 -13169.91808797]
541859.8376707988
Total parameters: 11
